In [1]:
##################
### Keagan Moo ###
### 2024/09/09 ###
### PC v Meta  ###
##################

# For supplamentary figures this script generate PCs for ATAC and RNA data and tries to correlate these with 
# metadata to see if any outstanding categories show up and give reviewers and idea of metadata's effect on 
# our analysis
suppressPackageStartupMessages(library("tidyverse"))
suppressPackageStartupMessages(library("Seurat"))
suppressPackageStartupMessages(library("glue"))
suppressPackageStartupMessages(library("SingleCellExperiment"))
suppressPackageStartupMessages(library("data.table"))
suppressPackageStartupMessages(library("Matrix"))
suppressPackageStartupMessages(library("ggplot2"))
suppressPackageStartupMessages(library("GGally"))
suppressPackageStartupMessages(library("ggdist"))
suppressPackageStartupMessages(library("pheatmap"))
suppressPackageStartupMessages(library("Matrix.utils"))

Warning message:
“package ‘ggdist’ was built under R version 4.4.3”


In [2]:
# RNA First
cDex <- "Type_1"
sca <- readRDS(file = glue("/nfs/turbo/umms-scjp/kgmoo/data/rna_sample_counts/{cDex}_rna_postDecontx_postArushiQC_PostTPM_1in10nTotal_CDR_sca.rds"))

# Arushi indicated this file will give us the nuclei that should be included in the analysis after qc
metadataFactors <- read.table("/nfs/turbo/umms-scjp/kgmoo/data/cluster_info_qc.tsv", sep = "\t", header = TRUE)
batchdataFactors <- read.table("/nfs/turbo/umms-scjp/kgmoo/data/batch-sample-info-full.tsv", sep = "\t", header = TRUE)
metadataFactorsRNA <- metadataFactors[(metadataFactors$modality == "rna") & (metadataFactors$demux_cat != "multiome"),]
metadataFactorsRNA <- merge(metadataFactorsRNA, batchdataFactors, by = "SNG.1ST")
metadataFactorsRNA <- metadataFactorsRNA[metadataFactorsRNA$coarse_cluster_name == cDex,]

In [3]:
# ATAC Second
small <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/data/{cDex}_FUSIONsnATAC_sce.rds"))
keepRows <- rowSums(counts(small) >= 1) >= ncol(counts(small))*0.1
small <- small[keepRows,]
metadataFactorsATAC <- metadataFactors[(metadataFactors$modality == "atac") & (metadataFactors$demux_cat != "multiome"),]
metadataFactorsATAC <- merge(metadataFactorsATAC, batchdataFactors, by = "SNG.1ST")
metadataFactorsATAC <- metadataFactorsATAC[metadataFactorsATAC$coarse_cluster_name == cDex,]

In [4]:
metadataFactorsRNA_UMAP <- metadataFactorsRNA %>% select(index, UMAP_1, UMAP_2, age, modality)
metadataFactorsATAC_UMAP <- metadataFactorsATAC %>% select(index, UMAP_1, UMAP_2, age, modality)
UMAPFrame <- rbind(metadataFactorsRNA_UMAP, metadataFactorsATAC_UMAP)

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/{cDex}_AgeUMAP.pdf"), width=6, height=6)
mygg <- ggplot(UMAPFrame, aes(x = UMAP_1, y = UMAP_2, color = age)) + 
        geom_point(alpha = 0.1) + theme_bw() + 
    scale_colour_gradientn(colours=rainbow(4))
mygg
dev.off()

pdf 
  2

,index,UMAP_1,UMAP_2,age,modality
,<chr>,<dbl>,<dbl>,<dbl>,<chr>
12,rna.510.NM.1.AGCGCCATCTCCAATT,-7.04478255,1.0210370,66,rna
24,rna.510.NM.1.TACTTACCAGTTAGGG,-1.59700659,2.7092559,66,rna
33,rna.510.NM.3.AGTACCACAATGCAGG,-0.82277050,3.1772522,66,rna
34,rna.510.NM.2.AAGTCGTAGTGCTCAT,-4.19318953,4.4464907,66,rna
45,rna.510.NM.1.AGTACTGCATTCATCT,-0.03621515,1.9931900,66,rna
47,rna.510.NM.4.TCGCACTGTCAAGGCA,-4.71346655,0.3961826,66,rna


In [12]:
head(metadataFactors)
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/REUMAP.pdf"), width=6, height=6)
mygg <- ggplot(metadataFactors, aes(x = UMAP_1, y = UMAP_2, color = coarse_cluster_name)) + 
        geom_point(alpha = 0.1) + theme_bw()
mygg
dev.off()

,index,UMAP_1,UMAP_2,modality,library,barcode,batch,cluster,demux_cat,SNG.1ST,hqaa_umi,fraction_mitochondrial,cohort,sample,tss_enrichment,log10_hqaa_umi,coarse_cluster_name
,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,rna.3172.NM.1.AAACAGCCAAGCCACT,2.4207619,5.4962825,rna,3172-NM-1,AAACAGCCAAGCCACT,3172,0,multiome,multiome,2974,0.008721516,multiome,multiome--3172,NA,7.997663,Type_1
2,rna.3172.NM.1.AAACAGCCAATAACCT,0.8719831,2.1690523,rna,3172-NM-1,AAACAGCCAATAACCT,3172,2,multiome,multiome,2278,0.001986456,multiome,multiome--3172,NA,7.731053,Type_2x
3,rna.3172.NM.1.AAACAGCCACTGGCTG,3.2940706,15.9974536,rna,3172-NM-1,AAACAGCCACTGGCTG,3172,0,multiome,multiome,8366,0.001031909,multiome,multiome--3172,NA,9.031931,Type_1
4,rna.3172.NM.1.AAACAGCCAGCAAATA,-3.4878534,-16.3451634,rna,3172-NM-1,AAACAGCCAGCAAATA,3172,4,multiome,multiome,8473,0.001188520,multiome,multiome--3172,NA,9.044640,Mesenchymal_Stem_Cell
5,rna.3172.NM.1.AAACAGCCAGTCTATG,-4.3422512,0.9228687,rna,3172-NM-1,AAACAGCCAGTCTATG,3172,2,multiome,multiome,5839,0.002525548,multiome,multiome--3172,NA,8.672315,Type_2x
6,rna.3172.NM.1.AAACAGCCATAGCGAG,-4.6229319,-4.8338571,rna,3172-NM-1,AAACAGCCATAGCGAG,3172,1,multiome,multiome,15595,0.000750333,multiome,multiome--3172,NA,9.654706,Type_2a


pdf 
  2

In [12]:
nTypes <- rev(c("Type_1", "Type_2a", "Type_2x", 
            "Endothelial", "FAP", "Smooth_Muscle", 
            "T_cell", "Neuronal", "Neuromuscular_junction", "Muscle_Fiber_Mixed", 
            "Satellite_Cell", "Adipocyte", "Macrophage"))

nColors <- rev(c("#90D9EA", "#68D0C4", "#007DBE", 
            "#FF999D", "#52389E", "#FFBD6F", 
            "#FA2D18", "#FDE854", "#B0DB8D", "#3BA545", 
            "#CBB9DC", "#929D9B", "#BD5522"))

#metadataFactorsBard <- metadataFactors[metadataFactors$coarse_cluster_name == "Mesenchymal_Stem_Cell",]$coarse_cluster_name <- "FAP"
BarFrame <- metadataFactors %>% select(index, modality, sample, coarse_cluster_name)
BarFrame <- BarFrame[BarFrame$sample != "multiome--3127",]
plotFrame <- table(BarFrame$coarse_cluster_name)
plotFrame <- data.frame(t(plotFrame/sum(plotFrame)))
colnames(plotFrame) <- c("NA", "Cluster", "Percent")
plotFrame$Cluster <- factor(plotFrame$Cluster, levels = nTypes)
plotFrame[order(plotFrame$Cluster), ]

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PerBard_Down.pdf"), width=7, height=6)
mygg <- ggplot(plotFrame, aes(x = Percent*100, y = Cluster, fill = "#DDDDDD"), position = position_stack(reverse = TRUE)) +
  geom_bar(position = 'dodge', stat = "identity", color = "black") +
  xlab("Percentage of nuclei") + ylab("Cluster") + ggtitle(glue("")) + 
  scale_fill_manual(values = "#DDDDDD") + 
  theme_bw() + theme(text=element_text(size=24), legend.position="none")
mygg
dev.off()

,NA,Cluster,Percent
,<fct>,<fct>,<dbl>
4,A,Macrophage,0.008094029
1,A,Adipocyte,0.010387848
8,A,Satellite_Cell,0.014866048
5,A,Muscle_Fiber_Mixed,0.015632113
6,A,Neuromuscular_junction,0.023605761
7,A,Neuronal,0.024940904
10,A,T_cell,0.034256260
9,A,Smooth_Muscle,0.037355542
3,A,FAP,0.049317107


pdf 
  2

In [5]:
colnames(metadataFactorsATAC)
colnames(metadataFactorsRNA)
groupsRNA

[1] "sample.y"      "age"           "sex"           "batch.x"      
[5] "bmi"           "ogtt"          "fins"          "Tissue.amount"
[9] "area_name"

[1] "sample.y"      "age"           "sex"           "batch.x"      
[5] "bmi"           "ogtt"          "fins"          "Tissue.amount"
[9] "area_name"

[1] "12055--499"  "12086--499"  "12074--499"  "32028--499"  "32063--499" 
    [6] "32104--499"  "32084--499"  "32001--499"  "12040--499"  "32063--499" 
   [11] "12044--499"  "32063--499"  "22019--499"  "12055--499"  "32054--499" 
   [16] "12165--499"  "12180--499"  "32054--499"  "32088--499"  "32052--499" 
   [21] "32074--499"  "12055--499"  "32074--499"  "12040--499"  "22029--499" 
   [26] "12136--499"  "32088--499"  "32037--499"  "12163--499"  "12136--499" 
   [31] "32074--499"  "32075--499"  "32001--499"  "32056--499"  "12052--499" 
   [36] "12180--499"  "12131--499"  "12099--499"  "22019--499"  "12086--499" 
   [41] "32048--499"  "32084--499"  "12052--499"  "32074--499"  "32054--499" 
   [46] "32074--499"  "32104--499"  "12055--499"  "32084--499"  "12180--499" 
   [51] "12044--499"  "12011--499"  "32052--499"  "12052--499"  "12131--499" 
   [56] "32074--499"  "12011--499"  "12165--499"  "32054--499"  "32104--499" 
   [61] "12086--499"  "12055--499"  "32054--499"  "12138--499"  "32114--499" 
   [66] "12052--499"  "12138--499"  "32063--499"  "32084--499"  "32001--499" 
   [71] "32048--499"  "12040--499"  "22029--499"  "22029--499"  "22019--499" 
   [76] "32063--499"  "32063--499"  "32114--499"  "32074--499"  "32040--499" 
   [81] "32054--499"  "12136--499"  "12055--499"  "32048--499"  "32001--499" 
   [86] "22019--499"  "32054--499"  "32054--499"  "12180--499"  "12165--499" 
   [91] "12055--499"  "12138--499"  "32084--499"  "12009--499"  "32029--499" 
   [96] "32073--499"  "32088--499"  "32029--499"  "32037--499"  "32104--499" 
  [101] "32074--499"  "32028--499"  "32088--499"  "32063--499"  "12055--499" 
  [106] "32048--499"  "32063--499"  "32028--499"  "32074--499"  "12055--499" 
  [111] "12040--499"  "32104--499"  "32048--499"  "32040--499"  "32074--499" 
  [116] "22029--499"  "12131--499"  "32088--499"  "12055--499"  "32052--499" 
  [121] "32001--499"  "12044--499"  "32048--499"  "32114--499"  "22029--499" 
  [126] "32074--499"  "12086--499"  "12040--499"  "12055--499"  "12180--499" 
  [131] "22029--499"  "32056--499"  "12138--499"  "32084--499"  "32088--499" 
  [136] "12044--499"  "12138--499"  "32054--499"  "32028--499"  "22029--499" 
  [141] "32073--499"  "12086--499"  "32029--499"  "32028--499"  "32075--499" 
  [146] "32054--499"  "22008--499"  "12040--499"  "12011--499"  "32074--499" 
  [151] "32088--499"  "32054--499"  "12131--499"  "12163--499"  "32063--499" 
  [156] "12074--499"  "12009--499"  "32063--499"  "12044--499"  "32056--499" 
  [161] "32114--499"  "22029--499"  "12011--499"  "32029--499"  "32084--499" 
  [166] "32029--499"  "12086--499"  "32048--499"  "32075--499"  "32104--499" 
  [171] "32084--499"  "32074--499"  "32054--499"  "12163--499"  "32001--499" 
  [176] "12074--499"  "32048--499"  "12011--499"  "32084--499"  "32056--499" 
  [181] "12163--499"  "12138--499"  "32052--499"  "32054--499"  "32054--499" 
  [186] "32028--499"  "12055--499"  "32074--499"  "32056--499"  "32088--499" 
  [191] "32063--499"  "32073--499"  "12131--499"  "12086--499"  "22019--499" 
  [196] "32056--499"  "12136--499"  "32048--499"  "32028--499"  "32074--499" 
  [201] "32052--499"  "12086--499"  "32054--499"  "12131--499"  "32029--499" 
  [206] "32063--499"  "32029--499"  "32063--499"  "32063--499"  "32084--499" 
  [211] "12099--499"  "32114--499"  "32084--499"  "32056--499"  "32084--499" 
  [216] "12052--499"  "12099--499"  "32001--499"  "32074--499"  "22019--499" 
  [221] "32084--499"  "32074--499"  "12040--499"  "12044--499"  "12165--499" 
  [226] "32054--499"  "12099--499"  "12165--499"  "12074--499"  "12055--499" 
  [231] "12011--499"  "12009--499"  "12099--499"  "32114--499"  "32029--499" 
  [236] "32075--499"  "32052--499"  "12055--499"  "32063--499"  "32084--499" 
  [241] "12086--499"  "32052--499"  "12055--499"  "12011--499"  "32073--499" 
  [246] "12165--499"  "32052--499"  "12165--499"  "12055--499"  "32029--499" 
  [251] "32029--499"  "32054--499"  "32029--499"  "12138--499"  "12055--499" 
  [256] "32074--499"  "320

In [4]:
groupsRNA <- colData(sca)[, c("sample")]
# Aggregate across cluster-sample groups
pbRNA <- aggregate.Matrix(t(counts(sca)), 
                       groupings = groupsRNA, fun = "sum")
metadataFactorsRNA <- metadataFactorsRNA %>% select("sample.y", "age", "sex", "batch.x", "bmi", "ogtt", "fins", "Tissue.amount", "area_name") %>% unique()

groupsATAC <- colData(small)[, c("sample")]
# Aggregate across cluster-sample groups
pbATAC <- aggregate.Matrix(t(counts(small)), 
                       groupings = groupsATAC, fun = "sum")
metadataFactorsATAC <- metadataFactorsATAC %>% select("sample.y", "age", "sex", "batch.x", "bmi", "ogtt", "fins", "Tissue.amount", "area_name") %>% unique()

Loading required package: MAST



In [7]:
head(metadataFactorsRNA)
dim(metadataFactorsRNA)

,SNG.1ST,index,UMAP_1,UMAP_2,modality,library,barcode,batch.x,cluster,demux_cat,⋯,clamp,sex,age,bmi,fins,ogtt,area_name,FAM,Tissue.amount,sample.y
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,⋯,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>
12,12001,rna.510.NM.1.AGCGCCATCTCCAATT,-7.04478255,1.0210370,rna,510-NM-1,AGCGCCATCTCCAATT,510,2,SNG-SNG,⋯,NA,M,66,30.37742,8.2,IGT,Kuopio,NA,9.4,12001--510
24,12001,rna.510.NM.1.TACTTACCAGTTAGGG,-1.59700659,2.7092559,rna,510-NM-1,TACTTACCAGTTAGGG,510,2,SNG-SNG,⋯,NA,M,66,30.37742,8.2,IGT,Kuopio,NA,9.4,12001--510
33,12001,rna.510.NM.3.AGTACCACAATGCAGG,-0.82277050,3.1772522,rna,510-NM-3,AGTACCACAATGCAGG,510,2,SNG-nS,⋯,NA,M,66,30.37742,8.2,IGT,Kuopio,NA,9.4,12001--510
34,12001,rna.510.NM.2.AAGTCGTAGTGCTCAT,-4.19318953,4.4464907,rna,510-NM-2,AAGTCGTAGTGCTCAT,510,2,SNG-SNG,⋯,NA,M,66,30.37742,8.2,IGT,Kuopio,NA,9.4,12001--510
45,12001,rna.510.NM.1.AGTACTGCATTCATCT,-0.03621515,1.9931900,rna,510-NM-1,AGTACTGCATTCATCT,510,2,SNG-nS,⋯,NA,M,66,30.37742,8.2,IGT,Kuopio,NA,9.4,12001--510
47,12001,rna.510.NM.4.TCGCACTGTCAAGGCA,-4.71346655,0.3961826,rna,510-NM-4,TCGCACTGTCAAGGCA,510,2,SNG-SNG,⋯,NA,M,66,30.37742,8.2,IGT,Kuopio,NA,9.4,12001--510


[1] 26137    30

In [19]:
# Thanks to https://www.cedricscherer.com/2021/06/06/visualizing-distributions-with-raincloud-plots-and-how-to-create-them-with-ggplot2/#rain

metadataFactorsRNA$batch.S <- 0
metadataFactorsRNA[metadataFactorsRNA$batch.x == "499",]$batch.S <- 1
metadataFactorsRNA[metadataFactorsRNA$batch.x == "510",]$batch.S <- 2
metadataFactorsRNA[metadataFactorsRNA$batch.x == "530",]$batch.S <- 3
metadataFactorsRNA[metadataFactorsRNA$batch.x == "551",]$batch.S <- 4
metadataFactorsRNA[metadataFactorsRNA$batch.x == "564",]$batch.S <- 5
metadataFactorsRNA[metadataFactorsRNA$batch.x == "567",]$batch.S <- 6
metadataFactorsRNA[metadataFactorsRNA$batch.x == "625",]$batch.S <- 7
metadataFactorsRNA[metadataFactorsRNA$batch.x == "647",]$batch.S <- 8
metadataFactorsRNA[metadataFactorsRNA$batch.x == "678",]$batch.S <- 9
metadataFactorsRNA[metadataFactorsRNA$batch.x == "692",]$batch.S <- 10

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/SampleAgeRaincloud.pdf"), width=6, height=6)
mygg <- ggplot(metadataFactorsRNA, aes(x = age, y = factor(batch.S, levels = c("1", "2", "3", "4", "5", "6", "7", "8", "9", "10")))) + 
  ## add half-violin from {ggdist} package
  ggdist::stat_halfeye(
    ## custom bandwidth
    adjust = .5, 
    ## adjust height
    width = .6, 
    ## move geom to the right
    justification = -.2, 
    ## remove slab interval
    .width = 0, 
    point_colour = NA
  ) + 
  geom_boxplot(
    width = .15, 
    ## remove outliers
    outlier.color = NA ## `outlier.shape = NA` or `outlier.alpha = 0` works as well
  ) +
  ## add dot plots from {ggdist} package
  ggdist::stat_dots(
    ## orientation to the left
    side = "left", 
    ## move geom to the left
    justification = 1.12, 
    ## adjust grouping (binning) of observations 
    binwidth = 0.25
  ) + theme_bw() + theme(text=element_text(size=18)) +
  xlab("Age (Years)") + ylab("Batch") + ggtitle(glue("")) 
mygg
dev.off()

pdf 
  2

pdf 
  2

In [6]:
# PCA
resultsRNA <- prcomp(pbRNA, scale = TRUE)
resultsATAC <- prcomp(pbATAC, scale = TRUE)

In [7]:
head(resultsATAC$x)
head(metadataFactorsRNA)
head(resultsATAC$sdev^2 / sum(resultsATAC$sdev^2))
colnames(metadataFactorsRNA)
#unique(metadataFactorsRNA$batch.x)
#unique(metadataFactorsRNA$hqaa_umi)
#unique(metadataFactorsRNA$fraction_mitochondrial)
#unique(metadataFactorsRNA$sample.x)
#unique(metadataFactorsRNA$sex)
#unique(metadataFactorsRNA$age)
#unique(metadataFactorsRNA$bmi)
#unique(metadataFactorsRNA$ogtt)
#unique(metadataFactorsRNA$Tissue.amount)
unique(metadataFactorsRNA$fins)
unique(metadataFactorsRNA$area_name)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,⋯,PC275,PC276,PC277,PC278,PC279,PC280,PC281,PC282,PC283,PC284
12001--510,-68.77201,-34.6915748,23.5663590,-23.053043,3.763498,-6.075155,-1.900938,10.8326920,-2.124697,13.3552634,⋯,-0.06612269,0.12423104,-0.152900695,0.01910354,-0.03868729,0.028184866,-0.013808268,4.772036e-03,-4.640856e-06,-1.312145e-14
12002--551,171.16920,-19.0018651,-8.4551635,4.242578,-7.070789,1.022978,-2.342398,-0.9587843,-1.657791,1.8559894,⋯,0.00428180,0.01466652,0.193633783,-0.16665361,0.19019929,-0.037925608,0.022219770,-8.776148e-03,-3.064809e-05,-2.631185e-14
12003--530,-143.11662,-0.2555288,36.4223302,-17.944600,-3.165160,30.451667,11.666810,7.9403882,-11.432512,12.7885280,⋯,0.01895219,0.12333620,-0.042904592,0.04520832,-0.03554496,0.006732722,-0.003687611,5.685937e-05,-1.004753e-05,-3.096655e-14
12004--530,-45.68151,-34.5027656,-39.0717974,17.362417,-7.173754,21.992309,-10.288372,-9.5191185,2.321451,-0.2707228,⋯,-0.03881691,0.01751412,-0.006836507,-0.02368550,0.03296043,-0.006401495,0.005014157,9.452665e-04,1.126897e-05,-3.845188e-14
12006--530,82.97645,10.2105911,-0.1044821,14.179034,-10.002788,-18.158615,10.622025,-9.7626968,20.926961,-4.0339112,⋯,-0.23814690,-0.01418203,0.269336436,-0.05875362,0.05319840,-0.022971059,0.014019410,-8.619948e-03,2.139127e-05,-7.618212e-14
12007--647,104.68012,19.4101026,-4.1000222,14.394215,-3.597050,-5.534634,8.140898,4.0601475,4.371788,-5.6574344,⋯,-0.27917147,-0.13815739,-0.105178229,-0.16517339,0.04118153,-0.001569945,0.012238865,-8.008208e-04,2.759493e-06,-4.185801e-14


,sample.y,age,sex,batch.x,bmi,ogtt,fins,Tissue.amount,area_name
,<chr>,<dbl>,<chr>,<int>,<dbl>,<chr>,<dbl>,<dbl>,<chr>
3,12001--510,66,M,510,30.37742,IGT,8.2,9.4,Kuopio
769,12002--551,61,M,551,25.10174,NGT,3.5,8.5,Kuopio
1047,12003--530,58,M,530,25.51226,NGT,4.2,6.3,Kuopio
2192,12004--530,64,M,530,23.03005,NGT,3.7,7.5,Kuopio
3036,12006--530,56,F,530,22.79218,NGT,6.4,7.8,Kuopio
3792,12007--647,64,F,647,21.35780,IGT,1.8,6.2,Kuopio


[1] 0.817225413 0.021839006 0.009301725 0.006477927 0.005759394 0.004445101

[1] "sample.y"      "age"           "sex"           "batch.x"      
[5] "bmi"           "ogtt"          "fins"          "Tissue.amount"
[9] "area_name"

[1]  8.2  3.5  4.2  3.7  6.4  1.8  7.1  4.1  7.9  6.3  4.5  3.4  6.7  7.6 10.5
 [16] 12.0  8.5  5.0  4.7 10.7 14.0  5.1  7.7  6.8  6.5  3.0  5.6 15.2  4.8 10.2
 [31]  2.5  9.9  9.3  9.1 13.2  3.8  8.9 12.8  5.4 12.6 19.2 10.8 11.2  9.0 49.5
 [46]  8.4  5.7 13.7 11.9 18.7  8.3 15.8  4.3  7.2 12.9 17.6 18.6 10.1 22.2  5.9
 [61] 11.0 20.7 12.5  3.1 10.9  7.4 12.4  4.6  5.8 10.6  4.0 19.7  5.5  3.9  3.2
 [76]  6.1  7.0  8.6 19.6  9.8 12.7  5.3 14.2  2.9 15.7  8.1 17.0 11.3  9.7 16.0
 [91]  6.9  1.4  2.7 10.3  7.5 15.4 18.4 14.5  7.3 14.6  3.6  4.4  9.5  2.8  6.0
[106]  4.9 14.7  8.8 11.7  9.2 12.3 36.4 22.4 25.9  5.2  3.3 13.3 22.6 16.1 11.5
[121]  6.6 17.3  2.4 10.0 14.3 13.4  9.4 25.3 18.1  7.8 14.1

[1] "Kuopio"      "Savitaipale" "Helsinki"

In [6]:
PlotDF <- data.frame(Vars <- (resultsRNA$sdev^2 / sum(resultsRNA$sdev^2)), PCs <- seq(1, length(Vars)))

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCVars_{cDex}_RNASamp.pdf"), width=6, height=6)
    mygg <- ggplot(data=PlotDF, aes(x=PCs, y=Vars)) +
        geom_point()+
        theme_bw() + #xlim(-0.07,0.07) + ylim(-0.07,0.07) + 
        xlab("PC#") + ylab("% Variance Explained") + ggtitle(glue("{cDex} % Variance of RNA PCs")) 
   print(mygg)
dev.off()

PlotDF <- data.frame(Vars <- (resultsATAC$sdev^2 / sum(resultsATAC$sdev^2)), PCs <- seq(1, length(Vars)))

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCVars_{cDex}_ATACSamp.pdf"), width=6, height=6)
    mygg <- ggplot(data=PlotDF, aes(x=PCs, y=Vars)) +
        geom_point()+
        theme_bw() + #xlim(-0.07,0.07) + ylim(-0.07,0.07) + 
        xlab("PC#") + ylab("% Variance Explained") + ggtitle(glue("{cDex} % Variance of ATAC PCs")) 
   print(mygg)
dev.off()

pdf 
  2

pdf 
  2

In [7]:
metadataUnpackRNA <- data.frame(sample <- metadataFactorsRNA$sample.y, batch510 <- metadataFactorsRNA$batch.x == "510", batch551 <- metadataFactorsRNA$batch.x == "551", batch530 <- metadataFactorsRNA$batch.x == "530", 
                   batch647 <- metadataFactorsRNA$batch.x == "647", batch499 <- metadataFactorsRNA$batch.x == "499", batch625 <- metadataFactorsRNA$batch.x == "625", batch567 <- metadataFactorsRNA$batch.x == "567", 
                   batch564 <- metadataFactorsRNA$batch.x == "564", batch678 <- metadataFactorsRNA$batch.x == "678", batch692 <- metadataFactorsRNA$batch.x == "692",
                   sex <- metadataFactorsRNA$sex == "F", age <- metadataFactorsRNA$age, bmi <- metadataFactorsRNA$bmi, ogttIGT <- metadataFactorsRNA$ogtt == "IGT", 
                   ogttNGT <- metadataFactorsRNA$ogtt == "NGT", ogttIFG <- metadataFactorsRNA$ogtt == "IFG",  ogttT2D <- metadataFactorsRNA$ogtt == "T2D", Tissue.amount <- metadataFactorsRNA$Tissue.amount,
                   fins <- metadataFactorsRNA$fins, area_nameKuopio <- metadataFactorsRNA$area_name == "Kuopio", area_nameSavitaipale <- metadataFactorsRNA$area_name == "Savitaipale",
                   area_nameHelsinki <- metadataFactorsRNA$area_name == "Helsinki")

colnames(metadataUnpackRNA) <- c("sample", "batch510", "batch551", "batch530", "batch647", "batch499", "batch625", "batch567", "batch564", "batch678", "batch692", 
                              "sex", "age", "bmi", "ogttIGT", "ogttNGT", "ogttIFG", "ogttT2D", "Tissue.amount", "fins", "area_nameKuopio", "area_nameSavitaipale", "area_nameHelsinki")
rownames(metadataUnpackRNA) <- metadataUnpackRNA$sample
sampleVecRNA <- metadataUnpackRNA$sample
metadataUnpackRNA$sample <- NULL
#metadataUnpack %>% unique(metadataUnpack)

metadataUnpackATAC <- data.frame(sample <- metadataFactorsATAC$sample.y, batch510 <- metadataFactorsATAC$batch.x == "510", batch551 <- metadataFactorsATAC$batch.x == "551", batch530 <- metadataFactorsATAC$batch.x == "530", 
                   batch647 <- metadataFactorsATAC$batch.x == "647", batch499 <- metadataFactorsATAC$batch.x == "499", batch625 <- metadataFactorsATAC$batch.x == "625", batch567 <- metadataFactorsATAC$batch.x == "567", 
                   batch564 <- metadataFactorsATAC$batch.x == "564", batch678 <- metadataFactorsATAC$batch.x == "678", batch692 <- metadataFactorsATAC$batch.x == "692",
                   sex <- metadataFactorsATAC$sex == "F", age <- metadataFactorsATAC$age, bmi <- metadataFactorsATAC$bmi, ogttIGT <- metadataFactorsATAC$ogtt == "IGT", 
                   ogttNGT <- metadataFactorsATAC$ogtt == "NGT", ogttIFG <- metadataFactorsATAC$ogtt == "IFG",  ogttT2D <- metadataFactorsATAC$ogtt == "T2D", Tissue.amount <- metadataFactorsATAC$Tissue.amount,
                   fins <- metadataFactorsATAC$fins, area_nameKuopio <- metadataFactorsATAC$area_name == "Kuopio", area_nameSavitaipale <- metadataFactorsATAC$area_name == "Savitaipale",
                   area_nameHelsinki <- metadataFactorsATAC$area_name == "Helsinki")

colnames(metadataUnpackATAC) <- c("sample", "batch510", "batch551", "batch530", "batch647", "batch499", "batch625", "batch567", "batch564", "batch678", "batch692", 
                              "sex", "age", "bmi", "ogttIGT", "ogttNGT", "ogttIFG", "ogttT2D", "Tissue.amount", "fins", "area_nameKuopio", "area_nameSavitaipale", 
                              "area_nameHelsinki")
rownames(metadataUnpackATAC) <- metadataUnpackATAC$sample
sampleVecATAC <- metadataUnpackATAC$sample
metadataUnpackATAC$sample <- NULL

In [54]:
head(metadataUnpack)
dim(metadataUnpack)

,batch510,batch551,batch530,batch647,batch499,batch625,batch567,batch564,batch678,batch692,⋯,bmi,ogttIGT,ogttNGT,ogttIFG,ogttT2D,Tissue.amount,fins,area_nameKuopio,area_nameSavitaipale,area_nameHelsinki
,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,⋯,<dbl>,<lgl>,<lgl>,<lgl>,<lgl>,<dbl>,<dbl>,<lgl>,<lgl>,<lgl>
12001--510,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,⋯,30.37742,TRUE,FALSE,FALSE,FALSE,9.4,8.2,TRUE,FALSE,FALSE
12002--551,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,⋯,25.10174,FALSE,TRUE,FALSE,FALSE,8.5,3.5,TRUE,FALSE,FALSE
12003--530,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,⋯,25.51226,FALSE,TRUE,FALSE,FALSE,6.3,4.2,TRUE,FALSE,FALSE
12004--530,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,⋯,23.03005,FALSE,TRUE,FALSE,FALSE,7.5,3.7,TRUE,FALSE,FALSE
12006--530,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,⋯,22.79218,FALSE,TRUE,FALSE,FALSE,7.8,6.4,TRUE,FALSE,FALSE
12007--647,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,⋯,21.35780,TRUE,FALSE,FALSE,FALSE,6.2,1.8,TRUE,FALSE,FALSE


[1] 287  22

In [10]:
namesTemp <- names(resultsRNA$x[,1])
PCVec <- data.frame(resultsRNA$x[,1])
rownames(PCVec) <- unique(groupsRNA)
metaVec <- data.frame(as.numeric(metadataUnpackRNA[,1]))
rownames(metaVec) <- rownames(metadataUnpackRNA)
dim(PCVec)
dim(metaVec)
corData <- merge(PCVec, metaVec, by = "row.names")
colnames(corData) <- c("id", "PC", "meta")
cor.test(corData$PC, corData$meta, method = "spearman")
cor.test(corData$PC, corData$meta, method = "spearman")[3]
cor.test(corData$PC, corData$meta, method = "spearman")[4]

[1] 287   1

[1] 287   1

Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”



	Spearman's rank correlation rho

data:  corData$PC and corData$meta
S = 3477769, p-value = 0.0471
alternative hypothesis: true rho is not equal to 0
sample estimates:
      rho 
0.1173032 


Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”


$p.value
[1] 0.04709667

Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”


$estimate
      rho 
0.1173032

In [8]:
HowManyPCs <- 25
stateHeatCoRNA <- data.frame(matrix(NA, nrow = HowManyPCs, ncol = ncol(metadataUnpackRNA)))
stateHeatPvRNA <- data.frame(matrix(NA, nrow = HowManyPCs, ncol = ncol(metadataUnpackRNA)))

for(pDex in 1:HowManyPCs) {
    PCVec <- data.frame(resultsRNA$x[,pDex])
    rownames(PCVec) <- unique(groupsRNA)
    for(mDex in 1:ncol(metadataUnpackRNA)) {
        metaVec <- data.frame(as.numeric(metadataUnpackRNA[,mDex]))
        rownames(metaVec) <- rownames(metadataUnpackRNA)
        corData <- merge(PCVec, metaVec, by = "row.names")
        colnames(corData) <- c("id", "PC", "meta")
        corTemp <- cor.test(corData$PC, corData$meta, method = "spearman")
        pTemp <- corTemp[3]
        cTemp <- corTemp[4]
        stateHeatCoRNA[pDex, mDex] <- cTemp
        stateHeatPvRNA[pDex, mDex] <- pTemp
        
    }
    
}

Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”


In [9]:
HowManyPCs <- 25
stateHeatCoATAC <- data.frame(matrix(NA, nrow = HowManyPCs, ncol = ncol(metadataUnpackATAC)))
stateHeatPvATAC <- data.frame(matrix(NA, nrow = HowManyPCs, ncol = ncol(metadataUnpackATAC)))

for(pDex in 1:HowManyPCs) {
    PCVec <- data.frame(resultsATAC$x[,pDex])
    rownames(PCVec) <- unique(groupsATAC)
    for(mDex in 1:ncol(metadataUnpackATAC)) {
        metaVec <- data.frame(as.numeric(metadataUnpackATAC[,mDex]))
        rownames(metaVec) <- rownames(metadataUnpackATAC)
        corData <- merge(PCVec, metaVec, by = "row.names")
        colnames(corData) <- c("id", "PC", "meta")
        corTemp <- cor.test(corData$PC, corData$meta, method = "spearman")
        pTemp <- corTemp[3]
        cTemp <- corTemp[4]
        stateHeatCoATAC[pDex, mDex] <- cTemp
        stateHeatPvATAC[pDex, mDex] <- pTemp
        
    }
    
}

Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”
Warning message in cor.test.default(corData$PC, corData$meta, method = "spearman"):
“Cannot compute exact p-value with ties”


In [10]:
colnames(stateHeatPvRNA) <- colnames(metadataUnpackRNA)
colnames(stateHeatCoRNA) <- colnames(metadataUnpackRNA)
rownames(stateHeatPvRNA) <- colnames(resultsRNA$x)[1:HowManyPCs]
rownames(stateHeatCoRNA) <- colnames(resultsRNA$x)[1:HowManyPCs]

colnames(stateHeatPvATAC) <- colnames(metadataUnpackATAC)
colnames(stateHeatCoATAC) <- colnames(metadataUnpackATAC)
rownames(stateHeatPvATAC) <- colnames(resultsATAC$x)[1:HowManyPCs]
rownames(stateHeatCoATAC) <- colnames(resultsATAC$x)[1:HowManyPCs]

In [11]:
col_palH <- c("#fee100","#ffe43d","#ffe75c","#ffea75","#ffee8e","#fff1a5","#fff4bb","#fff8d2","#fffbe8","#ffffff","#e2e3ea","#c6c7d5","#abadc0","#8f93ac","#757a98","#5b6185","#414a72","#25345f","#00204d")

stateHeatUntRNA <- as.matrix(stateHeatCoRNA)
boundColRNA <- round(max(c(abs(max(stateHeatUntRNA)), abs(min(stateHeatUntRNA)))), digits = 3)
brakersRNA <- seq(-boundColRNA, boundColRNA, by = ((2*boundColRNA)/(length(col_palH))))

stateHeatLabRNA <- as.matrix(stateHeatPvRNA)
stateHeatLabRNA[stateHeatLabRNA < 0.05] <- "*"
stateHeatLabRNA[stateHeatLabRNA >= 0.05] <- ""
stateHeatLabRNA <- stateHeatLabRNA[nrow(stateHeatLabRNA):1, ]
stateHeatUntRNA <- stateHeatUntRNA[nrow(stateHeatUntRNA):1, ]

stateHeatUntATAC <- as.matrix(stateHeatCoATAC)
boundColATAC <- round(max(c(abs(max(stateHeatUntATAC)), abs(min(stateHeatUntATAC)))), digits = 3)
brakersATAC <- seq(-boundColATAC, boundColATAC, by = ((2*boundColATAC)/(length(col_palH))))

stateHeatLabATAC <- as.matrix(stateHeatPvATAC)
stateHeatLabATAC[stateHeatLabATAC < 0.05] <- "*"
stateHeatLabATAC[stateHeatLabATAC >= 0.05] <- ""
stateHeatLabATAC <- stateHeatLabATAC[nrow(stateHeatLabATAC):1, ]
stateHeatUntATAC <- stateHeatUntATAC[nrow(stateHeatUntATAC):1, ]

In [16]:

head(stateHeatUnt)

,batch510,batch551,batch530,batch647,batch499,batch625,batch567,batch564,batch678,batch692,⋯,bmi,ogttIGT,ogttNGT,ogttIFG,ogttT2D,Tissue.amount,fins,area_nameKuopio,area_nameSavitaipale,area_nameHelsinki
PC25,-0.024407818,0.016585592,-0.06922682,-0.03146155,0.04699416,0.10453354,0.006194571,0.070013063,-0.037947627,-0.09091769,⋯,0.051393221,0.01906309,-0.06580247,0.043561971,0.0187641985,-0.035907353,0.0751683718,0.047459334,-0.01700047,-0.03641144
PC24,-0.026229296,0.027161911,0.04062268,0.08423576,-0.07795930,-0.12821427,-0.033998113,0.059064518,-0.006591841,0.07643685,⋯,-0.001611701,0.00733929,0.01007003,-0.017539425,-0.0042166738,0.041408432,-0.0201304736,-0.007080966,-0.05622298,0.04708525
PC23,0.005343005,0.002403709,-0.05901105,0.01387015,-0.03254376,0.05006784,-0.014117860,0.076351694,0.013896314,-0.03537725,⋯,0.033977709,-0.04479827,0.02777941,0.066718598,-0.0455400773,-0.085202984,-0.0163324195,0.010368558,0.05002995,-0.04605230
PC22,0.025014977,0.079923322,0.01466262,0.02418818,0.09228827,0.02486477,-0.068140285,0.009796066,-0.091216831,-0.20639782,⋯,-0.014299217,-0.04670457,0.03967246,0.004929381,-0.0010541685,-0.007000404,-0.0285814803,-0.014667716,-0.08439519,0.07480270
PC21,0.103945730,-0.087615190,-0.03401248,0.04042640,-0.04152972,0.15510882,-0.072750199,-0.027803541,0.021735261,-0.01503075,⋯,-0.101320935,-0.13172596,0.04488110,0.098816892,0.0003162505,-0.055748771,0.0005272043,0.012981772,0.02586500,-0.03159102
PC20,0.079780777,-0.003725749,-0.01069650,0.04262533,-0.06788045,0.00896485,-0.023481747,0.044514478,0.046677363,-0.12666154,⋯,0.047566763,-0.03650582,-0.01979283,0.101338901,-0.0287787989,-0.076250207,0.0903753103,0.001264458,0.01262892,-0.01024341


In [12]:
pheatmap(stateHeatUntRNA, 
                    col = rev(col_palH),
                    cluster_cols = FALSE,
                    breaks = brakersRNA,
                    cexCol = 1, 
                    cluster_rows = FALSE,
                    #cutree_rows = 2,
                    #treeheight_row = 0,
                    #cutree_cols = 2,
                    #treeheight_col = 16,
                    display_numbers = stateHeatLabRNA,
                    filename = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCxMetaHeat_{cDex}_RNASamp.pdf"),
                    #border_color=NA,
                    #show_colnames = F,
                    #main="Peak Set by Chromatin State Model Coefficient",
                    width = 8, height = 7, fontsize = 14
                    )

pheatmap(stateHeatUntATAC, 
                    col = rev(col_palH),
                    cluster_cols = FALSE,
                    breaks = brakersATAC,
                    cexCol = 1, 
                    cluster_rows = FALSE,
                    #cutree_rows = 2,
                    #treeheight_row = 0,
                    #cutree_cols = 2,
                    #treeheight_col = 16,
                    display_numbers = stateHeatLabATAC,
                    filename = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCxMetaHeat_{cDex}_ATACSamp.pdf"),
                    #border_color=NA,
                    #show_colnames = F,
                    #main="Peak Set by Chromatin State Model Coefficient",
                    width = 8, height = 7, fontsize = 14
                    )

In [12]:
resPlot <- results$x[,1:25]
rownames(resPlot) <- unique(groups)
rownames(metadataFactorsRNA) <- sampleVec
checkPlot <- merge(metadataFactorsRNA, resPlot, by = "row.names")
head(checkPlot)

,Row.names,sample.y,age,sex,batch.x,bmi,ogtt,fins,Tissue.amount,area_name,⋯,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24,PC25
,<I<chr>>,<chr>,<dbl>,<chr>,<int>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,12001--510,12001--510,66,M,510,30.37742,IGT,8.2,9.4,Kuopio,⋯,1.1018166,1.0911485,-1.0239771,-0.800393514,-0.1470563,0.4950781,-0.3378092,-0.6463285,0.3064519,-2.0531913
2,12002--551,12002--551,61,M,551,25.10174,NGT,3.5,8.5,Kuopio,⋯,0.3222838,-0.4843926,-0.3363228,0.008376754,-0.7731211,0.1732223,-0.2382866,0.1388272,0.1132789,0.3672104
3,12003--530,12003--530,58,M,530,25.51226,NGT,4.2,6.3,Kuopio,⋯,-2.2906402,0.0423677,1.8281184,-0.447346923,0.1298818,-0.4460949,0.1875170,0.5965054,-0.8734863,0.2100510
4,12004--530,12004--530,64,M,530,23.03005,NGT,3.7,7.5,Kuopio,⋯,-1.3754389,-0.2045202,2.4933877,-1.219862512,-0.5240926,2.9412112,0.2975717,1.4627565,3.4226679,-0.4470736
5,12006--530,12006--530,56,F,530,22.79218,NGT,6.4,7.8,Kuopio,⋯,-1.6438272,2.9874062,-0.8064357,0.807944580,-3.7141649,-0.0512595,0.9638012,1.5295659,0.5112253,0.3990924
6,12007--647,12007--647,64,F,647,21.35780,IGT,1.8,6.2,Kuopio,⋯,8.1111468,1.8336004,-1.4478629,0.009423995,1.5518599,-3.5307819,3.1825462,-6.4077519,2.3390526,-4.5207115


In [13]:
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCMetaCheck_PC1vBatch_{cDex}_Samp.pdf"), width=6, height=6)
    mygg <- ggplot(data=checkPlot, aes(x=factor(batch.x), y=PC1)) +
        geom_boxplot()+
        theme_bw() + #xlim(-0.07,0.07) + ylim(-0.07,0.07) + 
        xlab("Batch") + ylab("PC1")
   print(mygg)
dev.off()

pdf 
  2

In [16]:
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PC2vTissue.amount_{cDex}.pdf"), width=6, height=6)
    mygg <- ggplot(data=checkPlot, aes(x=PC2, y=Tissue.amount)) +
        geom_point(size = 0.5, alpha = 0.8)+
        theme_bw() + #xlim(-0.07,0.07) + ylim(-0.07,0.07) + 
        xlab("PC2") + ylab("Tissue.amount")
   print(mygg)
dev.off()

pdf 
  2

In [16]:
# Thanks Claude
# Limit to top n_components
pc_data <- results$x
n_components <- 10
  # Limit to top n_components
  n_comp <- min(n_components, ncol(pc_data))
  pc_data <- pc_data[, 1:n_comp, drop = FALSE]
  
  # Create all pairwise combinations
  pc_names <- colnames(pc_data)
  pairs_list <- list()
  
  for (i in 1:n_comp) {
    for (j in 1:n_comp) {
      pairs_list[[paste(pc_names[i], pc_names[j], sep = "_vs_")]] <- data.frame(
        PC_x = pc_data[, i],
        PC_y = pc_data[, j],
        PC_x_name = pc_names[i],
        PC_y_name = pc_names[j],
        stringsAsFactors = FALSE
      )
    }
  }
  
  # Combine all pairs into one dataframe
  plot_data <- do.call(rbind, pairs_list)
  
  # Create factor levels for proper ordering
  plot_data$PC_x_name <- factor(plot_data$PC_x_name, levels = pc_names)
  plot_data$PC_y_name <- factor(plot_data$PC_y_name, levels = rev(pc_names))
  
  # Calculate variance explained for axis labels
  var_explained <- round(100 * results$sdev^2 / sum(results$sdev^2), 1)
  
  # Create labels with variance explained
  pc_labels <- paste0(pc_names, " (", var_explained[1:n_comp], "%)")
  names(pc_labels) <- pc_names
  
  # Create the plot with upper triangle only
  p <- ggplot(plot_data, aes(x = PC_x, y = PC_y)) +
    geom_point(size = 0.5, alpha = 0.1) +
    facet_grid(PC_y_name ~ PC_x_name, 
               labeller = labeller(PC_x_name = pc_labels, PC_y_name = pc_labels),
               scales = "free") +
    theme_minimal() +
    theme(
      strip.text = element_text(size = 8),
      axis.text = element_text(size = 6),
      axis.title = element_blank(),
      panel.grid.minor = element_blank(),
      panel.spacing = unit(0.1, "lines")
    ) +
    labs(
      title = paste("Upper Triangle Plot of Top", n_comp, "Principal Components"),
      subtitle = "Diagonal shows distribution of each PC, upper triangle shows pairwise relationships"
    )
  
  # Add diagonal density plots
  diag_data <- plot_data[plot_data$PC_x_name == plot_data$PC_y_name, ]
  
  p <- p + 
    geom_density(data = diag_data, aes(x = PC_x), 
                 fill = "lightblue", alpha = 0.5, inherit.aes = FALSE) +
    geom_blank(data = diag_data, aes(x = PC_x, y = PC_y))

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCMatrix10_D_{cDex}.pdf"), width=10, height=10)
   print(p)
dev.off()


pdf 
  2

In [14]:
# Internet still win Claude, sorry
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCMatrix10_K_{cDex}_RNASamp.pdf"), width=10, height=10)
ggpairs(resultsRNA$x[, 1:10], 
        title = "Scatterplot",
        upper = list(continuous = "cor"),
        lower = list(continuous = wrap("smooth", alpha = 0.25, size = 0.5)))
dev.off()

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250908_kgmoo_hSMMeta/Plots/PCMatrix10_K_{cDex}_ATACSamp.pdf"), width=10, height=10)
ggpairs(resultsATAC$x[, 1:10], 
        title = "Scatterplot",
        upper = list(continuous = "cor"),
        lower = list(continuous = wrap("smooth", alpha = 0.25, size = 0.5)))
dev.off()

pdf 
  2

pdf 
  2